In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 95
==================================================
Week: 14 of 24
Day: 95 of 168
Date: May 11, 2026 
Topic: API Authentication & Security

Week 14 Progress:
✅ Day 92: Docker Fundamentals
✅ Day 93: Dockerize Security System
✅ Day 94: RESTful API Design & FastAPI Deep Dive
🔄 Day 95: API Authentication & Security (TODAY!)
⬜ Day 96: CI/CD & GitHub Actions
⬜ Day 97: Automated Testing & Deployment
⬜ Day 98: Model Monitoring & Logging

Progress: 42.9% (3/7 days)

==================================================
🎯 Week 14 Project: MLOps & Production Engineering
- Containerize ML projects with Docker
- Build production-grade APIs with FastAPI
- Automate deployments with GitHub Actions CI/CD
- Add monitoring, logging, and error tracking

🎯 Today's Learning Objectives:
1. Understand API security fundamentals and attack vectors
2. Implement API key authentication
3. Implement JWT token authentication with OAuth2
4. Add rate limiting and request throttling
5. Secure the AI Security System API endpoints

📚 Today's Structure:
   Part 1 (1.5h): API Security Fundamentals
   Part 2 (2h):   API Key & JWT Authentication
   Part 3 (1.5h): Rate Limiting & Request Throttling
   Part 4 (1h):   Securing the Security System API

🎯 SUCCESS CRITERIA:
   ✅ Understand common API attack vectors and how to prevent them
   ✅ Implement API key authentication with hashed key storage
   ✅ Implement JWT authentication with access + refresh tokens
   ✅ Add rate limiting per client to prevent abuse
   ✅ Apply authentication to the Security System API endpoints

==================================================
"""

In [1]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys
!{sys.executable} -m pip install python-jose[cryptography] passlib[bcrypt] python-multipart slowapi -q

print("✅ Libraries installed!")

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "=" * 80)
print("📚 IMPORTING LIBRARIES")
print("=" * 80)

import os
import time
import hashlib
import secrets
from datetime import datetime, timedelta
from typing import Optional

print("\n✅ All libraries imported successfully!")
print("=" * 80)

✅ Libraries installed!

📚 IMPORTING LIBRARIES

✅ All libraries imported successfully!


'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
print("\n" + "=" * 80)
print("🔐 PART 1: API SECURITY FUNDAMENTALS")
print("=" * 80)


🔐 PART 1: API SECURITY FUNDAMENTALS


In [3]:
# ==================================================
# EXERCISE 1.1: COMMON API ATTACK VECTORS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: Common API Attack Vectors")
print("=" * 80)

"""
📖 THEORY: Why API Security Matters

An unsecured ML API exposes you to:
- Unauthorized inference (someone using YOUR GPU for free)
- Model theft (extracting your model through repeated queries)
- Data poisoning (injecting bad inputs to corrupt results)
- Denial of Service (flooding the API with requests)
- Sensitive data leaks (if the model was trained on private data)
"""

print("""
📋 COMMON API ATTACK VECTORS & DEFENSES:

  ─── 1. Broken Authentication ────────────────────────────────
  Attack:   No auth, or weak tokens (short, predictable)
  Defense:  JWT with short expiry + refresh tokens
            API keys with 32+ random bytes
            Never send credentials in URL params

  ─── 2. Broken Object Level Authorization (BOLA) ─────────────
  Attack:   GET /api/v1/detections/123 → user sees OTHER user's data
  Defense:  Always filter by authenticated user ID
            Never trust client-supplied IDs blindly

  ─── 3. Excessive Data Exposure ──────────────────────────────
  Attack:   API returns full model config, internal paths, stack traces
  Defense:  Use response schemas (Pydantic) — only expose what you intend
            Custom exception handlers — never expose raw errors in prod

  ─── 4. Rate Limiting / DoS ──────────────────────────────────
  Attack:   1000 requests/second → GPU maxed out, real users blocked
  Defense:  slowapi rate limiter (per IP or per API key)
            Request queue with max concurrent inferences

  ─── 5. Injection ────────────────────────────────────────────
  Attack:   Malicious file uploads (not an image, triggers code execution)
  Defense:  Validate file type by CONTENT (magic bytes), not just extension
            Run file processing in isolated container

  ─── 6. Security Misconfiguration ────────────────────────────
  Attack:   CORS allows *, debug mode on in prod, secrets in code
  Defense:  Explicit CORS origins, ENV vars for secrets, no debug in prod

  ─── 7. Model Extraction ─────────────────────────────────────
  Attack:   Query API thousands of times to reconstruct the model
  Defense:  Rate limiting, confidence score rounding, watermarking outputs
""")

print("""
📋 AUTHENTICATION vs AUTHORIZATION:

  Authentication  → WHO are you?
                    (Verifying identity: login, API key, JWT)

  Authorization   → WHAT are you allowed to do?
                    (Permissions: read-only vs admin, own data vs all data)

  Both are required. Auth without Authz = everyone sees everything.
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: Common API Attack Vectors

📋 COMMON API ATTACK VECTORS & DEFENSES:

  ─── 1. Broken Authentication ────────────────────────────────
  Attack:   No auth, or weak tokens (short, predictable)
  Defense:  JWT with short expiry + refresh tokens
            API keys with 32+ random bytes
            Never send credentials in URL params

  ─── 2. Broken Object Level Authorization (BOLA) ─────────────
  Attack:   GET /api/v1/detections/123 → user sees OTHER user's data
  Defense:  Always filter by authenticated user ID
            Never trust client-supplied IDs blindly

  ─── 3. Excessive Data Exposure ──────────────────────────────
  Attack:   API returns full model config, internal paths, stack traces
  Defense:  Use response schemas (Pydantic) — only expose what you intend
            Custom exception handlers — never expose raw errors in prod

  ─── 4. Rate Limiting / DoS ──────────────────────────────────
  Attack:   1000 requests/second → GPU maxed out, real users blocked

In [4]:
# ==================================================
# EXERCISE 1.2: SECRETS MANAGEMENT
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Secrets Management Best Practices")
print("=" * 80)

"""
📖 THEORY: Never hardcode secrets

Secrets = API keys, JWT secret keys, DB passwords, Redis passwords.
These must NEVER appear in source code or be committed to git.
"""

print("""
📋 SECRETS MANAGEMENT RULES:

  ❌ NEVER do this:
     SECRET_KEY = "mysecretkey123"                  # hardcoded
     API_KEY = "sk-abc123"                           # hardcoded
     redis_url = "redis://:password@localhost:6379"  # hardcoded

  ✅ Always do this:
     SECRET_KEY = os.environ.get("SECRET_KEY")
     API_KEY = os.environ.get("API_KEY")
     REDIS_URL = os.environ.get("REDIS_URL")

  ✅ Generate strong secrets:
     # 32-byte random secret key (256 bits)
     python -c "import secrets; print(secrets.token_hex(32))"

  ✅ .env file for local development (git-ignored!):
     SECRET_KEY=a3f8c2d1e4b5a6f7c8d9e0a1b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6e7f8a9b0
     API_KEY=sk-a1b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6
     REDIS_URL=redis://localhost:6379/0

  ✅ In production (Docker/cloud):
     Pass secrets as environment variables in docker-compose.yml
     Or use a secrets manager (AWS Secrets Manager, HashiCorp Vault)
     NEVER put real secrets in docker-compose.yml — use .env file

  ✅ .gitignore must include:
     .env
     .env.*
     *.key
     secrets/
""")

# Demonstrate secure key generation
print("⏱️  Generating example secure keys:")
print(f"   JWT Secret Key  : {secrets.token_hex(32)}")
print(f"   API Key         : sk-{secrets.token_hex(16)}")
print(f"   Redis Password  : {secrets.token_urlsafe(24)}")
print("\n   (These are freshly generated examples — use your own!)")

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Secrets Management Best Practices

📋 SECRETS MANAGEMENT RULES:

  ❌ NEVER do this:
     SECRET_KEY = "mysecretkey123"                  # hardcoded
     API_KEY = "sk-abc123"                           # hardcoded
     redis_url = "redis://:password@localhost:6379"  # hardcoded

  ✅ Always do this:
     SECRET_KEY = os.environ.get("SECRET_KEY")
     API_KEY = os.environ.get("API_KEY")
     REDIS_URL = os.environ.get("REDIS_URL")

  ✅ Generate strong secrets:
     # 32-byte random secret key (256 bits)
     python -c "import secrets; print(secrets.token_hex(32))"

  ✅ .env file for local development (git-ignored!):
     SECRET_KEY=a3f8c2d1e4b5a6f7c8d9e0a1b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6e7f8a9b0
     API_KEY=sk-a1b2c3d4e5f6a7b8c9d0e1f2a3b4c5d6
     REDIS_URL=redis://localhost:6379/0

  ✅ In production (Docker/cloud):
     Pass secrets as environment variables in docker-compose.yml
     Or use a secrets manager (AWS Secrets Manager, HashiCorp Vault)
     NEVER put real secrets 

In [5]:
print("\n" + "=" * 80)
print("🔑 PART 2: API KEY & JWT AUTHENTICATION")
print("=" * 80)


🔑 PART 2: API KEY & JWT AUTHENTICATION


In [6]:
# ==================================================
# EXERCISE 2.1: API KEY AUTHENTICATION
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: API Key Authentication")
print("=" * 80)

"""
📖 THEORY: API Keys

API keys are simple tokens passed in request headers.
Best for: server-to-server communication, simple integrations.
Not ideal for: user-facing apps (can't expire easily, no user identity).

Security best practices:
- Generate with cryptographically secure random bytes (32+)
- Store HASHED in DB (like passwords — never store plaintext)
- Pass in header (X-API-Key), never in URL params
- Prefix keys for easy identification (sk- = secret key)
"""

api_key_auth_py = '''
# ============================================================
# api/auth/api_key.py — API Key Authentication
# ============================================================

import hashlib
import secrets
from fastapi import Security, HTTPException, status
from fastapi.security import APIKeyHeader
from api.core.config import settings

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def hash_api_key(api_key: str) -> str:
    """Hash an API key for storage — never store plaintext keys."""
    return hashlib.sha256(api_key.encode()).hexdigest()


def generate_api_key() -> tuple[str, str]:
    """
    Generate a new API key.
    Returns: (plaintext_key, hashed_key)
    Store hashed_key in DB. Give plaintext_key to the user ONCE.
    """
    key = f"sk-{secrets.token_hex(32)}"
    return key, hash_api_key(key)


async def verify_api_key(api_key: str = Security(api_key_header)):
    """
    FastAPI dependency — verifies API key from X-API-Key header.
    Hash the incoming key and compare to stored hash.
    """
    if not api_key:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Missing API key. Include X-API-Key header.",
        )

    # Hash the incoming key and compare to stored hash
    incoming_hash = hash_api_key(api_key)

    # In production: look up hash in database
    # Here: compare against env var for simplicity
    expected_hash = hash_api_key(settings.API_KEY)

    if not secrets.compare_digest(incoming_hash, expected_hash):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid API key.",
        )

    return api_key


# ── Usage in routes ───────────────────────────────────────────
# @router.post("/detections")
# async def run_detection(
#     file: UploadFile = File(...),
#     api_key: str = Security(verify_api_key),   # ← protects this endpoint
# ):
#     ...

# ── Usage in Streamlit (calling the API) ──────────────────────
# headers = {"X-API-Key": os.environ.get("API_KEY")}
# response = requests.post(f"{API_URL}/api/v1/detections",
#                          files={"file": frame},
#                          headers=headers)
'''

print("📄 api/auth/api_key.py:")
print(api_key_auth_py)

print("""
💡 WHY secrets.compare_digest()?

  Regular string comparison (==) is vulnerable to timing attacks:
  - Attacker measures response time to guess characters one by one
  - "sk-a..." vs "sk-b..." → different time → leaks information

  secrets.compare_digest() always takes the SAME time regardless of
  where strings differ → timing attack impossible ✅
""")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: API Key Authentication
📄 api/auth/api_key.py:

# ============================================================
# api/auth/api_key.py — API Key Authentication
# ============================================================

import hashlib
import secrets
from fastapi import Security, HTTPException, status
from fastapi.security import APIKeyHeader
from api.core.config import settings

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def hash_api_key(api_key: str) -> str:
    """Hash an API key for storage — never store plaintext keys."""
    return hashlib.sha256(api_key.encode()).hexdigest()


def generate_api_key() -> tuple[str, str]:
    """
    Generate a new API key.
    Returns: (plaintext_key, hashed_key)
    Store hashed_key in DB. Give plaintext_key to the user ONCE.
    """
    key = f"sk-{secrets.token_hex(32)}"
    return key, hash_api_key(key)


async def verify_api_key(api_key: str = Security(api_key_header)):
    """
    FastAPI dependency — 

In [7]:
# ==================================================
# EXERCISE 2.2: JWT AUTHENTICATION
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: JWT Token Authentication")
print("=" * 80)

"""
📖 THEORY: JWT (JSON Web Tokens)

JWT = a self-contained token that carries user identity + claims.
Structure: header.payload.signature (three base64 parts separated by dots)

  header    → algorithm used (HS256)
  payload   → claims: user_id, role, expiry (exp), issued-at (iat)
  signature → HMAC of header+payload using SECRET_KEY

Why JWT for ML APIs:
- Stateless: server doesn't store sessions (scales horizontally)
- Self-contained: user info is IN the token (no DB lookup per request)
- Expirable: short-lived access tokens + longer refresh tokens

Flow:
  1. User POSTs credentials → /auth/login
  2. Server verifies, returns access_token (15min) + refresh_token (7d)
  3. Client includes access_token in Authorization: Bearer <token> header
  4. When access_token expires, client POSTs refresh_token → /auth/refresh
  5. Server returns new access_token (refresh_token stays valid)
"""

jwt_auth_py = '''
# ============================================================
# api/auth/jwt.py — JWT Authentication
# ============================================================

from datetime import datetime, timedelta, timezone
from typing import Optional
from jose import JWTError, jwt
from passlib.context import CryptContext
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from pydantic import BaseModel
from api.core.config import settings

# ── Password Hashing ──────────────────────────────────────────
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str) -> str:
    return pwd_context.hash(password)

def verify_password(plain: str, hashed: str) -> bool:
    return pwd_context.verify(plain, hashed)


# ── Token Schemas ─────────────────────────────────────────────
class Token(BaseModel):
    access_token: str
    refresh_token: str
    token_type: str = "bearer"

class TokenData(BaseModel):
    user_id: str
    role: str = "viewer"


# ── Token Creation ────────────────────────────────────────────
def create_access_token(user_id: str, role: str = "viewer") -> str:
    """Create a short-lived access token (15 minutes)."""
    payload = {
        "sub": user_id,
        "role": role,
        "type": "access",
        "exp": datetime.now(timezone.utc) + timedelta(minutes=15),
        "iat": datetime.now(timezone.utc),
    }
    return jwt.encode(payload, settings.SECRET_KEY, algorithm="HS256")


def create_refresh_token(user_id: str) -> str:
    """Create a long-lived refresh token (7 days)."""
    payload = {
        "sub": user_id,
        "type": "refresh",
        "exp": datetime.now(timezone.utc) + timedelta(days=7),
        "iat": datetime.now(timezone.utc),
    }
    return jwt.encode(payload, settings.SECRET_KEY, algorithm="HS256")


# ── Token Verification ────────────────────────────────────────
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/login")

async def get_current_user(token: str = Depends(oauth2_scheme)) -> TokenData:
    """FastAPI dependency — decode and verify JWT access token."""
    credentials_exception = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Could not validate credentials",
        headers={"WWW-Authenticate": "Bearer"},
    )
    try:
        payload = jwt.decode(token, settings.SECRET_KEY, algorithms=["HS256"])

        user_id = payload.get("sub")
        token_type = payload.get("type")

        if not user_id or token_type != "access":
            raise credentials_exception

        return TokenData(user_id=user_id, role=payload.get("role", "viewer"))

    except JWTError:
        raise credentials_exception


# ── Role-Based Authorization ──────────────────────────────────
def require_role(required_role: str):
    """
    Dependency factory for role-based access control.
    Usage: Depends(require_role("admin"))
    """
    async def check_role(current_user: TokenData = Depends(get_current_user)):
        role_hierarchy = {"viewer": 0, "operator": 1, "admin": 2}
        user_level = role_hierarchy.get(current_user.role, 0)
        required_level = role_hierarchy.get(required_role, 99)

        if user_level < required_level:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Requires {required_role} role. You have: {current_user.role}"
            )
        return current_user
    return check_role


# ── Auth Router ───────────────────────────────────────────────
# @router.post("/auth/login", response_model=Token)
# async def login(form_data: OAuth2PasswordRequestForm = Depends()):
#     user = await get_user(form_data.username)
#     if not user or not verify_password(form_data.password, user.hashed_password):
#         raise HTTPException(status_code=401, detail="Invalid credentials")
#
#     return Token(
#         access_token=create_access_token(user.id, user.role),
#         refresh_token=create_refresh_token(user.id),
#     )

# ── Usage in protected routes ─────────────────────────────────
# @router.delete("/detections/{id}")
# async def delete_detection(
#     id: str,
#     current_user: TokenData = Depends(require_role("admin")),  # admin only
# ):
#     ...
#
# @router.get("/detections")
# async def list_detections(
#     current_user: TokenData = Depends(get_current_user),  # any logged-in user
# ):
#     ...
'''

print("📄 api/auth/jwt.py:")
print(jwt_auth_py)

print("""
📊 ACCESS TOKEN vs REFRESH TOKEN:

  Access Token:
  ├── Expiry: 15 minutes
  ├── Sent with: every API request (Authorization: Bearer <token>)
  ├── If stolen: attacker has 15 min max
  └── Stored: in memory (not localStorage — XSS risk)

  Refresh Token:
  ├── Expiry: 7 days
  ├── Sent with: only to /auth/refresh endpoint
  ├── If stolen: revoke it in DB (maintain a blocklist)
  └── Stored: httpOnly cookie (not accessible to JS — XSS safe)
""")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: JWT Token Authentication
📄 api/auth/jwt.py:

# ============================================================
# api/auth/jwt.py — JWT Authentication
# ============================================================

from datetime import datetime, timedelta, timezone
from typing import Optional
from jose import JWTError, jwt
from passlib.context import CryptContext
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from pydantic import BaseModel
from api.core.config import settings

# ── Password Hashing ──────────────────────────────────────────
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str) -> str:
    return pwd_context.hash(password)

def verify_password(plain: str, hashed: str) -> bool:
    return pwd_context.verify(plain, hashed)


# ── Token Schemas ─────────────────────────────────────────────
class Token(BaseModel):
    access_token: str
    refresh_token: str
    to

In [8]:
print("\n" + "=" * 80)
print("⏱️  PART 3: RATE LIMITING & REQUEST THROTTLING")
print("=" * 80)


⏱️  PART 3: RATE LIMITING & REQUEST THROTTLING


In [9]:
# ==================================================
# EXERCISE 3.1: RATE LIMITING WITH SLOWAPI
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Rate Limiting with SlowAPI")
print("=" * 80)

"""
📖 THEORY: Rate Limiting

Rate limiting restricts how many requests a client can make
in a given time window. Critical for ML APIs because:
- GPU inference is expensive — prevent abuse
- Protect against DoS attacks
- Ensure fair usage across all clients
- Prevent model extraction attacks
"""

rate_limiting_py = '''
# ============================================================
# api/core/rate_limiter.py — Rate Limiting with SlowAPI
# ============================================================

from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
from fastapi import Request, Response

# ── Initialize Limiter ────────────────────────────────────────
# key_func: what identifies a client (IP address here)
# For authenticated APIs: use API key or user ID instead of IP
limiter = Limiter(key_func=get_remote_address)


# ── Custom Key Function (per API key) ─────────────────────────
def get_api_key_or_ip(request: Request) -> str:
    """
    Rate limit by API key if present, otherwise by IP.
    Authenticated clients get their own rate limit bucket.
    """
    api_key = request.headers.get("X-API-Key")
    if api_key:
        return f"apikey:{api_key[:8]}"   # Use first 8 chars as bucket key
    return get_remote_address(request)

authenticated_limiter = Limiter(key_func=get_api_key_or_ip)


# ── Register in main.py ───────────────────────────────────────
# app.state.limiter = limiter
# app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)


# ── Usage in routes ───────────────────────────────────────────
# from slowapi import Limiter
# from api.core.rate_limiter import limiter

# @router.post("/detections")
# @limiter.limit("10/minute")        # 10 requests per minute per IP
# async def run_detection(request: Request, ...):
#     ...

# @router.get("/detections")
# @limiter.limit("60/minute")        # Reads are cheaper — higher limit
# async def list_detections(request: Request, ...):
#     ...
'''

print("📄 api/core/rate_limiter.py:")
print(rate_limiting_py)

print("""
📋 RATE LIMIT STRATEGY FOR ML APIS:

  Endpoint                  Limit           Reason
  ──────────────────────────────────────────────────────────────
  POST /detections          10/minute       GPU inference — expensive
  GET  /detections          60/minute       Reads — cheap
  GET  /detections/{id}     120/minute      Single record lookup
  POST /auth/login          5/minute        Prevent brute force
  GET  /health              300/minute      Allow frequent health checks

  Different limits per role:
  ├── viewer    → 5 inferences/minute
  ├── operator  → 30 inferences/minute
  └── admin     → 100 inferences/minute

  Rate limit headers returned to client:
  X-RateLimit-Limit:     10
  X-RateLimit-Remaining: 7
  X-RateLimit-Reset:     1715432100   (Unix timestamp when window resets)

  When limit exceeded → 429 Too Many Requests
""")

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Rate Limiting with SlowAPI
📄 api/core/rate_limiter.py:

# ============================================================
# api/core/rate_limiter.py — Rate Limiting with SlowAPI
# ============================================================

from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
from fastapi import Request, Response

# ── Initialize Limiter ────────────────────────────────────────
# key_func: what identifies a client (IP address here)
# For authenticated APIs: use API key or user ID instead of IP
limiter = Limiter(key_func=get_remote_address)


# ── Custom Key Function (per API key) ─────────────────────────
def get_api_key_or_ip(request: Request) -> str:
    """
    Rate limit by API key if present, otherwise by IP.
    Authenticated clients get their own rate limit bucket.
    """
    api_key = request.headers.get("X-API-Key")
    if api_key:
        return f"apike

In [10]:
# ==================================================
# EXERCISE 3.2: EXCEPTION HANDLERS
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.2: Custom Exception Handlers")
print("=" * 80)

"""
📖 THEORY: Exception Handlers

Never expose raw Python tracebacks in production.
Custom exception handlers let you:
- Return consistent, clean error responses
- Log errors server-side with full detail
- Hide internal implementation details from clients
"""

exception_handlers_py = '''
# ============================================================
# api/core/exceptions.py — Custom Exception Handlers
# ============================================================

from fastapi import FastAPI, Request, status
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import ValidationError
from loguru import logger


# ── Custom Exception Classes ──────────────────────────────────
class ModelNotLoadedException(Exception):
    pass

class DetectionFailedException(Exception):
    def __init__(self, detail: str):
        self.detail = detail


# ── Exception Handlers ────────────────────────────────────────
def register_exception_handlers(app: FastAPI):
    """Register all custom exception handlers on the app."""

    @app.exception_handler(RequestValidationError)
    async def validation_exception_handler(
        request: Request, exc: RequestValidationError
    ):
        """Return clean 422 errors for Pydantic validation failures."""
        errors = []
        for error in exc.errors():
            errors.append({
                "field": " → ".join(str(loc) for loc in error["loc"]),
                "message": error["msg"],
                "type": error["type"],
            })
        return JSONResponse(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            content={"detail": "Validation error", "errors": errors},
        )

    @app.exception_handler(ModelNotLoadedException)
    async def model_not_loaded_handler(request: Request, exc):
        logger.warning("Request received but model not loaded yet")
        return JSONResponse(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            content={"detail": "Model is still loading. Try again in a moment."},
        )

    @app.exception_handler(DetectionFailedException)
    async def detection_failed_handler(request: Request, exc):
        logger.error(f"Detection failed: {exc.detail}")
        return JSONResponse(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            content={"detail": exc.detail},
        )

    @app.exception_handler(Exception)
    async def generic_exception_handler(request: Request, exc: Exception):
        """
        Catch-all handler — logs full traceback server-side,
        returns generic message to client (never expose internals).
        """
        logger.exception(f"Unhandled exception: {exc}")
        return JSONResponse(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            content={"detail": "An internal error occurred."},
        )


# ── Register in main.py ───────────────────────────────────────
# from api.core.exceptions import register_exception_handlers
# register_exception_handlers(app)
'''

print("📄 api/core/exceptions.py:")
print(exception_handlers_py)

print("""
💡 KEY PRINCIPLE:

  Development:  return full error details (traceback, field names, values)
  Production:   log full details SERVER-SIDE, return generic message to client

  In FastAPI, control this with:
  app = FastAPI(debug=False)   # Never True in production

  And always use the catch-all exception handler as a safety net.
""")

print("\n✅ Exercise 3.2 Complete!")
print("=" * 80)


EXERCISE 3.2: Custom Exception Handlers
📄 api/core/exceptions.py:

# ============================================================
# api/core/exceptions.py — Custom Exception Handlers
# ============================================================

from fastapi import FastAPI, Request, status
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import ValidationError
from loguru import logger


# ── Custom Exception Classes ──────────────────────────────────
class ModelNotLoadedException(Exception):
    pass

class DetectionFailedException(Exception):
    def __init__(self, detail: str):
        self.detail = detail


# ── Exception Handlers ────────────────────────────────────────
def register_exception_handlers(app: FastAPI):
    """Register all custom exception handlers on the app."""

    @app.exception_handler(RequestValidationError)
    async def validation_exception_handler(
        request: Request, exc: RequestValidation

In [11]:
print("\n" + "=" * 80)
print("🔒 PART 4: SECURING THE SECURITY SYSTEM API")
print("=" * 80)


🔒 PART 4: SECURING THE SECURITY SYSTEM API


In [12]:
# ==================================================
# EXERCISE 4.1: APPLYING AUTH TO THE SECURITY SYSTEM
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.1: Applying Auth to the Security System API")
print("=" * 80)

print("""
📂 Updated Security System API structure with auth:

  api/
  ├── main.py                    ← Register exception handlers + rate limiter
  ├── routers/
  │   ├── auth.py                ← NEW: /auth/login, /auth/refresh, /auth/logout
  │   ├── detections.py          ← UPDATED: protected with JWT + rate limited
  │   ├── cameras.py             ← UPDATED: protected with JWT + rate limited
  │   └── models.py              ← UPDATED: admin-only endpoints
  ├── auth/
  │   ├── __init__.py
  │   ├── api_key.py             ← API key auth (for server-to-server)
  │   └── jwt.py                 ← JWT auth (for user-facing)
  └── core/
      ├── rate_limiter.py        ← NEW: SlowAPI setup
      └── exceptions.py          ← NEW: Custom exception handlers
""")

security_matrix_py = '''
# ============================================================
# Security Matrix — Who can access what
# ============================================================

"""
  Endpoint                    Auth Required   Role        Rate Limit
  ──────────────────────────────────────────────────────────────────
  GET  /health                No              -           300/min
  GET  /                      No              -           300/min

  POST /auth/login            No              -           5/min
  POST /auth/refresh          Yes (refresh)   any         10/min
  POST /auth/logout           Yes (access)    any         10/min

  GET  /api/v1/detections     Yes (JWT)       viewer+     60/min
  POST /api/v1/detections     Yes (JWT)       operator+   10/min
  GET  /api/v1/detections/{id}Yes (JWT)       viewer+     120/min
  DEL  /api/v1/detections/{id}Yes (JWT)       admin       30/min

  GET  /api/v1/cameras        Yes (JWT)       viewer+     60/min
  POST /api/v1/cameras        Yes (JWT)       admin       10/min
  DEL  /api/v1/cameras/{id}   Yes (JWT)       admin       10/min

  GET  /api/v1/models         Yes (JWT)       viewer+     30/min
  POST /api/v1/models/switch  Yes (JWT)       admin       5/min

  All endpoints also accept X-API-Key header as alternative to JWT
  (for Streamlit UI → FastAPI server-to-server communication)
"""

# ── Updated detection router with full security ───────────────

from fastapi import APIRouter, Depends, File, UploadFile, Request
from api.auth.jwt import get_current_user, require_role, TokenData
from api.core.rate_limiter import limiter

router = APIRouter()

@router.get("/detections")
@limiter.limit("60/minute")
async def list_detections(
    request: Request,                                    # Required by SlowAPI
    current_user: TokenData = Depends(get_current_user), # Any logged-in user
):
    ...

@router.post("/detections", status_code=201)
@limiter.limit("10/minute")
async def run_detection(
    request: Request,
    file: UploadFile = File(...),
    current_user: TokenData = Depends(require_role("operator")), # operator+
):
    ...

@router.delete("/detections/{detection_id}", status_code=204)
@limiter.limit("30/minute")
async def delete_detection(
    request: Request,
    detection_id: str,
    current_user: TokenData = Depends(require_role("admin")),    # admin only
):
    ...
'''

print("📄 Security matrix + updated detection router:")
print(security_matrix_py)

print("\n✅ Exercise 4.1 Complete!")
print("=" * 80)


EXERCISE 4.1: Applying Auth to the Security System API

📂 Updated Security System API structure with auth:

  api/
  ├── main.py                    ← Register exception handlers + rate limiter
  ├── routers/
  │   ├── auth.py                ← NEW: /auth/login, /auth/refresh, /auth/logout
  │   ├── detections.py          ← UPDATED: protected with JWT + rate limited
  │   ├── cameras.py             ← UPDATED: protected with JWT + rate limited
  │   └── models.py              ← UPDATED: admin-only endpoints
  ├── auth/
  │   ├── __init__.py
  │   ├── api_key.py             ← API key auth (for server-to-server)
  │   └── jwt.py                 ← JWT auth (for user-facing)
  └── core/
      ├── rate_limiter.py        ← NEW: SlowAPI setup
      └── exceptions.py          ← NEW: Custom exception handlers

📄 Security matrix + updated detection router:

# ============================================================
# Security Matrix — Who can access what
# =====================================

In [13]:
# ==================================================
# EXERCISE 4.2: DAY 95 SUMMARY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 4.2: Day 95 Summary")
print("=" * 80)

print("""
📚 WHAT WE LEARNED TODAY:

✅ API Security Fundamentals:
   • 7 common API attack vectors: broken auth, BOLA, excessive exposure,
     DoS, injection, misconfiguration, model extraction
   • Authentication (who are you?) vs Authorization (what can you do?)
   • Secrets management: env vars, .env files, never hardcode

✅ API Key Authentication:
   • Generate with secrets.token_hex(32) — cryptographically secure
   • Store HASHED with SHA-256 (never plaintext)
   • Pass in X-API-Key header (never in URL)
   • secrets.compare_digest() prevents timing attacks

✅ JWT Authentication:
   • JWT structure: header.payload.signature
   • Access tokens: 15 min expiry, sent with every request
   • Refresh tokens: 7 day expiry, only sent to /auth/refresh
   • python-jose for encoding/decoding, passlib/bcrypt for passwords
   • Role-based authorization: viewer → operator → admin hierarchy

✅ Rate Limiting:
   • SlowAPI decorators: @limiter.limit("10/minute")
   • Rate limit by IP for anonymous, by API key for authenticated
   • Different limits per endpoint based on compute cost
   • 429 Too Many Requests with rate limit headers

✅ Exception Handling:
   • Never expose raw tracebacks in production
   • Custom handlers for validation, model errors, catch-all
   • Log full details server-side, return generic messages to client

✅ Security System API:
   • Defined full security matrix: endpoint → auth → role → rate limit
   • API key auth for Streamlit → FastAPI server-to-server calls
   • JWT auth for user-facing access with 3-tier role system

💡 KEY INSIGHTS:
   1. Hash API keys before storing — treat them like passwords
   2. Short-lived access tokens + refresh tokens = security + UX balance
   3. Rate limit by API key not IP for authenticated APIs
      (multiple users behind same corporate IP share one bucket otherwise)
   4. Exception handlers are your last line of defense against data leaks
""")

print("\n✅ Exercise 4.2 Complete!")
print("=" * 80)


EXERCISE 4.2: Day 95 Summary

📚 WHAT WE LEARNED TODAY:

✅ API Security Fundamentals:
   • 7 common API attack vectors: broken auth, BOLA, excessive exposure,
     DoS, injection, misconfiguration, model extraction
   • Authentication (who are you?) vs Authorization (what can you do?)
   • Secrets management: env vars, .env files, never hardcode

✅ API Key Authentication:
   • Generate with secrets.token_hex(32) — cryptographically secure
   • Store HASHED with SHA-256 (never plaintext)
   • Pass in X-API-Key header (never in URL)
   • secrets.compare_digest() prevents timing attacks

✅ JWT Authentication:
   • JWT structure: header.payload.signature
   • Access tokens: 15 min expiry, sent with every request
   • Refresh tokens: 7 day expiry, only sent to /auth/refresh
   • python-jose for encoding/decoding, passlib/bcrypt for passwords
   • Role-based authorization: viewer → operator → admin hierarchy

✅ Rate Limiting:
   • SlowAPI decorators: @limiter.limit("10/minute")
   • Rate lim

In [14]:
print("\n" + "=" * 80)
print("🎯 DAY 95 COMPLETE! ✅")
print("=" * 80)

print("""
OBJECTIVES ACHIEVED:
   ✅ Understood common API attack vectors and defenses
   ✅ Implemented API key auth with hashed storage and timing-safe comparison
   ✅ Implemented JWT auth with access + refresh tokens and role hierarchy
   ✅ Added SlowAPI rate limiting per endpoint with per-key buckets
   ✅ Built custom exception handlers for clean production error responses
   ✅ Applied full security matrix to the Security System API

💡 KEY FILES CREATED TODAY:
   api/
   ├── auth/
   │   ├── api_key.py       ← API key generation, hashing, verification
   │   └── jwt.py           ← JWT creation, verification, role-based auth
   ├── routers/
   │   └── auth.py          ← /auth/login, /auth/refresh, /auth/logout
   └── core/
       ├── rate_limiter.py  ← SlowAPI setup and key functions
       └── exceptions.py    ← Custom exception handlers

🎯 TOMORROW (DAY 96):
   - CI/CD fundamentals
   - GitHub Actions workflows
   - Automated testing pipeline
   - Docker build and push to registry on merge
   - Deploy to cloud on successful CI

Week 14 Progress: 57.1% (4/7 days complete)
Overall Progress: Day 95/168 (56.5% complete)
""")

print("=" * 80)


🎯 DAY 95 COMPLETE! ✅

OBJECTIVES ACHIEVED:
   ✅ Understood common API attack vectors and defenses
   ✅ Implemented API key auth with hashed storage and timing-safe comparison
   ✅ Implemented JWT auth with access + refresh tokens and role hierarchy
   ✅ Added SlowAPI rate limiting per endpoint with per-key buckets
   ✅ Built custom exception handlers for clean production error responses
   ✅ Applied full security matrix to the Security System API

💡 KEY FILES CREATED TODAY:
   api/
   ├── auth/
   │   ├── api_key.py       ← API key generation, hashing, verification
   │   └── jwt.py           ← JWT creation, verification, role-based auth
   ├── routers/
   │   └── auth.py          ← /auth/login, /auth/refresh, /auth/logout
   └── core/
       ├── rate_limiter.py  ← SlowAPI setup and key functions
       └── exceptions.py    ← Custom exception handlers

🎯 TOMORROW (DAY 96):
   - CI/CD fundamentals
   - GitHub Actions workflows
   - Automated testing pipeline
   - Docker build and push 